# Display features

In [ ]:
import pickle

with open(r"C:\Users\bilel.guetarni\Desktop\SEQ-RT\artix.pkl", "rb") as f:
    artix = pickle.load(f)

with open(r"C:\Users\bilel.guetarni\Desktop\SEQ-RT\hnscc.pkl", "rb") as f:
    hnscc = pickle.load(f)

In [ ]:
for p in hnscc:
    for ct in p.ct:
        if ct.rtdose is None:
            print(f"ct {ct.path} has no RTDOSE")

        if ct.rtstruct is None:
            print(f"ct {ct.path} has no RTSTRUCT")

In [ ]:
p.ct[0].rtdose.path

In [ ]:
import pandas
import re

def transform_(x):
    if isinstance(x, str):
        digits = re.findall("\d", x)
        if len(digits) == 0:
            return None
        else:
            return int(digits[0])
    else:
        return None

def retrieve_xero_info(p):
    cm = pandas.DataFrame(p.clinical_measurements)
    cm = cm[(cm["type"] == "AE") & (cm["sample"] == "XEROSTOMIE")]
    cm["value"] = cm["value"].apply(transform_)
    cm.loc[(cm["visitID"] == "S0"), "visitID"] = "Inclusion"
    if (cm[(cm["visitID"] == "Inclusion") & (cm["sample"] == "XEROSTOMIE")]["value"].astype('Int64') > 0).any():
        # exclude patient because xerstomia baseline present
        xerostomia = None
    else:
        try:
            cm = cm[cm["visitID"] == "M6"]
            xerostomia = int(cm["value"].values[0])
        except (IndexError, ValueError):
            xerostomia = None
    return xerostomia

xero = list(map(retrieve_xero_info, artix))
stats = {k: xero.count(k) for k in set(xero)}
print("artix:")
for j, c in stats.items():
    print(f"\t {j}: {c} ({int(100*c/len(xero))}%)")

In [ ]:
def retrieve_xero_info(p):
    try:
        xerostomia = int(p.clinical['Xerostomia Grade'])
    except ValueError:
        xerostomia = None
    return xerostomia

xero = list(map(retrieve_xero_info, hnscc))
stats = {k: xero.count(k) for k in set(xero)}
print("hnscc:")
for j, c in stats.items():
    print(f"\t {j}: {c} ({int(100*c/len(xero))}%)")

# Fit

### original
parotid_gland_ipsi
parotid_gland_contra
submandibular_gland_ipsi
submandibular_gland_contra
mandible
oral_cavity

### total segmentator
parotid_gland_left
parotid_gland_right
submandibular_gland_right
submandibular_gland_left
superior_pharyngeal_constrictor
middle_pharyngeal_constrictor
inferior_pharyngeal_constrictor
mandible
esophagus

In [ ]:
import plotly.express as px
from sklearn.decomposition import PCA

def pca_visualization(x, y):
    x = PCA(n_components=2, random_state=0).fit_transform(x)
    fig = px.scatter(x=x[:,0], y=x[:,1], color=y)
    fig.show()

# Analysis

## radiomics

In [ ]:
import os
import pandas
import plotly.express as px

path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments"

for exp in os.listdir(path_):
    result_df = []
    for run in os.listdir(os.path.join(path_, exp)):
        with open(os.path.join(path_, exp, run, "params.json"), "r") as f:
            params = json.load(f)

        metrics_df = pandas.read_csv(os.path.join(path_, exp, run, "metrics.csv"))
        metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
        avg_metrics = metrics_df.mean().to_dict()
        for k, v in avg_metrics.items():
            try:
                oars = params["oars"]
            except KeyError:
                oars =  params["oar"]

            try:
                features = tuple(params["features"].items())
            except AttributeError:
                features = params["features"]
                features = {k: len(list(filter(lambda i: i[0] == k, features))) for k in set(map(lambda i: i[0], features))}
                features = tuple(features.items())
            
            result_df.append({"exp": exp, "run": run, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "oars": oars, "features": features})
    result_df = pandas.DataFrame(result_df)
    fig = px.bar(result_df, x="run", y="value", color="metric", barmode="group", hover_data=['features', 'oars'], title=exp)
    fig.update_yaxes(range=[0,1])
    # fig.update_layout(autosize=False, width=1400, height=2000)
    fig.show()

In [ ]:
import os, json
import pandas
import plotly.express as px

path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\005"
result_df = []
for run in os.listdir(path_):
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

    metrics_df = pandas.read_csv(os.path.join(path_, run, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"oar": params["oar"], "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

result_df = pandas.DataFrame(result_df)

fig = px.bar(result_df, x="oar", y="value", color="metric", barmode="group")
fig.update_layout(autosize=False, width=800, height=400)
fig.show()
# fig.write_image(r"C:\Users\bilel.guetarni\Downloads\newplot.png")

In [ ]:
import os, json
import pandas
import plotly.express as px


path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\002"
result_df = []
for run in os.listdir(path_):
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

    metrics_df = pandas.read_csv(os.path.join(path_, run, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"oar": params["oar"], "metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "run": run.split("_")[-1]})

result_df = pandas.DataFrame(result_df)

for run in result_df["run"].unique():
    print(run)
    fig = px.bar(result_df[result_df["run"] == run], x="oar", y="value", color="metric", barmode="group")
    fig.update_layout(autosize=False, width=800, height=400)
    fig.show()
    fig.write_image(r"C:\Users\bilel.guetarni\Downloads\newplot_{}.png".format(run))

In [ ]:


import os, json
import pandas
import plotly.express as px

run_names = {1: "shape", 2: "first order", 3: "texture", 4: "radiomics", 5: "radiomics+dosiomics"}



path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\002"
result_df = []
for run in os.listdir(path_):
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

    metrics_df = pandas.read_csv(os.path.join(path_, run, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"oar": params["oar"], "metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "run": run_names[int(run.split("_")[-1])]})

result_df = pandas.DataFrame(result_df)

fig = px.bar(result_df, x="oar", y="value", facet_row="metric", barmode="group", facet_col="run")
fig.update_layout(autosize=False, width=1200, height=400)

# # Get unique x values (categories)
# x_vals = result_df["oar"].unique()

# # Add vertical dashed lines *between* categories
# for i in range(len(x_vals) - 1):
#     fig.add_vline(
#         x=i + 0.5,  # halfway between category i and i+1
#         line=dict(color="black", width=2, dash="dash"),
#         row="all", col="all"  # apply to all facet subplots
#     )

# fig.show()
fig.write_image(r"C:\Users\bilel.guetarni\Downloads\newplot.svg", width=1400, height=1200)

In [ ]:
# result_df.to_csv(r"C:\Users\bilel.guetarni\Desktop\tmp\metrics.csv")
# result_df = result_df.drop(columns=["std"])
# result_df = result_df.set_index(["oar", "run"])
result_df

In [ ]:
# Pivot
table = result_df.pivot(index=["oar", "run"], columns="metric", values="value")

# Flatten the index so oar and run are normal columns
table = table.reset_index()

table.to_csv(r"C:\Users\bilel.guetarni\Desktop\tmp\metrics.csv")

In [ ]:
import os, json
import pandas
import plotly.express as px

result_df = []
path_ = r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\007"
for run in os.listdir(path_):
    # print(run)
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

        # fts = params["features"]["dvh"]
        # if isinstance(fts, list) and len(fts) == 0:
        #     continue

        # print(run)
        # print(params["oars"])
        # print(params["features"])

    metrics_df = pandas.read_csv(os.path.join(path_, run, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "run": run, "oars": params["oars"], "features": tuple(params["features"].items())})

fig = px.bar(result_df, x="run", y="value", color="metric", facet_row="metric", barmode="group", hover_data=['features', 'oars'])
fig.update_yaxes(range=[0, 1])
fig.update_layout(autosize=False, width=1200, height=800)
fig.show()

## 008

In [ ]:
import os, glob, json
import tqdm

def filter_features(features_dict):
    return "+".join([i for i in features_dict.keys() if features_dict[i] == -1])

result_df = []
for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\008_*\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)
        oars = "+".join(params["oars"])
        features = filter_features(params["features"])
        if params["feature_reduction_N"]:
            dim = params["feature_reduction_N"]
        else:
            dim = len(params["features_ordered"])

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()

    for k, v in avg_metrics.items():
        result_df.append({"metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "exp": os.path.split(exp)[1],
                          "oars": oars, "features": features, "dim": dim})

In [ ]:
import pandas
import plotly.express as px

df = pandas.DataFrame(result_df)
df = df[(df["metric"] == "f1_score") & (df["dim"] == 100)]

fn_ = lambda i: 1 if "deepnn" in i else 0
df["deepnn"] = df["features"].map(fn_)

sorted_features = sorted(df["features"].unique(), key= lambda i: len(i.split("+")))
print(sorted_features)

print(df["oars"].unique())

fig = px.bar(df, x="oars", y="value", color="oars", facet_row="features", facet_row_spacing=0.01,
            barmode="group", hover_data=['features', 'oars'], category_orders={"features": sorted_features})
fig.update_yaxes(range=[0, 1])
fig.update_layout(autosize=False, width=1000, height=8000)
fig.show()

In [ ]:
import itertools
import pandas
import plotly.express as px
import os, glob, json
import tqdm

N = 100

result_df = []
for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\008_000\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)
        oars = params["oars"]
        if len(oars) > 1:
            continue

        features = [i for i in params["features"].keys() if params["features"][i] == -1]
        if len(features) > 1:
            continue

        if params["feature_reduction_N"]:
            # dim = params["feature_reduction_N"]
            continue
        else:
            dim = len(params["features_ordered"])

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()

    result_df.append({"auc": avg_metrics["auc"], "metrics": avg_metrics, "oars": oars, "features": features, "dim": dim})

print(len(result_df))
result_df = sorted(result_df, key=lambda i: i["auc"], reverse=True)[:N]
print(len(result_df))
result_df = [[{"metric": k, "value": v, "oars": i["oars"], "features": i["features"], "dim": i["dim"]} for k, v in i["metrics"].items()] for i in result_df]
result_df = list(itertools.chain.from_iterable(result_df))
df = pandas.DataFrame(result_df)
print(len(df))

In [ ]:
import copy

dff = copy.deepcopy(df)
dff["oars"] = dff["oars"].map(lambda i: "+".join(i))
dff["features"] = dff["features"].map(lambda i: "+".join(i))

# fig = px.bar(dff, x="oars", y="value", color="features", barmode="group", hover_data=['features', 'oars', "dim"], facet_row="metric", facet_col="dim", category_orders={"dim": sorted(dff["dim"].unique())})
# fig.update_layout(xaxis=dict(categoryorder="total descending"))
# fig.update_yaxes(range=[0, 1])
# fig.update_layout(height=2000)
# fig.show()


# dff = dff[dff["dim"] == 10]
for m in dff["metric"].unique():
    fig = px.bar(dff[dff["metric"] == m], x="oars", y="value", color="features", barmode="group", hover_data=['features', 'oars', "dim"], title=m)
    # fig.update_layout(xaxis=dict(categoryorder="total descending"))
    fig.update_xaxes(categoryarray=sorted(dff["oars"].unique()))
    fig.update_yaxes(range=[0, 1])
    fig.update_layout(height=400)
    fig.show()

## dim

In [ ]:
import pandas
import plotly.express as px
import os, glob, json
import tqdm


result_df = []
for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\008_000\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)
        if len(params["oars"]) > 1:
            continue
        oars = params["oars"][0]

        features = [i for i in params["features"].keys() if params["features"][i] == -1]
        if len(features) > 1:
            continue
        features = features[0]

        if params["feature_reduction_N"]:
            dim = params["feature_reduction_N"]
        else:
            dim = len(params["features_ordered"])

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        result_df.append({"metric": k, "value": v, "std": metrics_df.std().to_dict()[k], "oars": oars, "features": features, "dim": dim})

df = pandas.DataFrame(result_df)
print(len(df))

In [ ]:
for m in df["metric"].unique():
    fig = px.line(df[df["metric"] == m].sort_values("dim", axis=0), x="dim", y="value", color='features', facet_col="oars", title=m)
    # fig.update_yaxes(range=[0, 1])
    fig.update_layout(width=1500)
    fig.show()

## top 100 features OARs

In [ ]:
import itertools
import pandas
import plotly.express as px
import os, glob, json
import tqdm

N = 100

result_df = []
for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\008_000\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)
        oars = params["oars"]
        features = [i for i in params["features"].keys() if params["features"][i] == -1]

        if params["feature_reduction_N"]:
            dim = params["feature_reduction_N"]
        else:
            dim = len(params["features_ordered"])

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()

    result_df.append({"oars": oars, "features": features, "dim": dim, **avg_metrics})

print(len(result_df))
result_df = sorted(result_df, key=lambda i: i["auc"], reverse=True)[:N]
print(len(result_df))

In [ ]:
def count_key(df, key, col):
    count = map(lambda i: 1 if key in i else 0, [i[col] for i in df])
    return sum(count)

def get_uniques(df, col):
    return set(itertools.chain.from_iterable([i[col] for i in df]))

oars = {i: count_key(result_df, i, "oars") for i in get_uniques(result_df, "oars")}
print(oars)
features = {i: count_key(result_df, i, "features") for i in get_uniques(result_df, "features")}
print(features)

## mean dose + (radiomics,DL)

In [ ]:
import pandas
import plotly.express as px
import os, glob, json
import tqdm


result_df = []

for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\008_000\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)

        oars = params["oars"]
        if len(oars) > 1:
            continue

        features = [i for i in params["features"].keys() if params["features"][i]]
        if len(features) > 1:
            continue
        if not(any([i in features for i in ["radiomics", "deepnn"]])):
            continue

        if params["feature_reduction_N"]:
            continue

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    std = metrics_df.std().to_dict()
    for k, v in  metrics_df.mean().to_dict().items():
        result_df.append({"std": std[k], "metric": k, "value": v, "oars": oars, "features": features, "mean dose": "no"})

for exp in tqdm.tqdm(glob.glob(r"C:\Users\bilel.guetarni\Desktop\ARTIX\experiments\010_000\*")):
    with open(os.path.join(exp, "params.json"), "r") as f:
        params = json.load(f)

        oars = params["oars"]
        if len(oars) > 1:
            continue

        features = [i for i in params["features"].keys() if params["features"][i]]
        if not(any([i in features for i in ["radiomics", "deepnn"]])):
            continue
        features.remove("dvh")

        if params["feature_reduction_N"]:
            continue

    metrics_df = pandas.read_csv(os.path.join(exp, "metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    std = metrics_df.std().to_dict()
    for k, v in  metrics_df.mean().to_dict().items():
        result_df.append({"std": std[k], "metric": k, "value": v, "oars": oars, "features": features, "mean dose": "yes"})

df = pandas.DataFrame(result_df)
print(df)

In [ ]:
import copy

dff = copy.deepcopy(df)
dff["oars"] = dff["oars"].map(lambda i: "+".join(i))
dff["features"] = dff["features"].map(lambda i: "+".join(i))
for m in dff["metric"].unique():
    fig = px.bar(dff[dff["metric"] == m], error_y="std", x="oars", y="value", color="mean dose", barmode="group", hover_data=['features', 'oars'], facet_col="features", title=m)
    fig.update_xaxes(categoryarray=sorted(dff["oars"].unique()))
    fig.update_yaxes(range=[0, 1])
    fig.update_layout(height=600)
    fig.show()

## 014

In [ ]:
import os, json
import pandas
import plotly.express as px

path_ = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments\014"



result_df = []
for run in os.listdir(path_):
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

    metrics_df = pandas.read_csv(os.path.join(path_, run, "internal_metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        oar = params["oars"] if isinstance(params["oars"], str) else "+".join(params["oars"])
        features = "+".join([k for k,v in params["features"].items() if v])
        result_df.append({"cohort": "internal", "oar": oar.split("_")[0], "features": features, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

    metrics_df = pandas.read_csv(os.path.join(path_, run, "external_metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        oar = params["oars"] if isinstance(params["oars"], str) else "+".join(params["oars"])
        features = "+".join([k for k,v in params["features"].items() if v])
        result_df.append({"cohort": "external", "oar": oar.split("_")[0], "features": features, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

result_df = pandas.DataFrame(result_df)
fig = px.bar(result_df, x="features", y="value", color="metric", barmode="group", facet_col="oar", facet_row="cohort")
# fig.update_layout(autosize=False, width=1600, height=900)
fig.show()

## 015

In [ ]:
import os, json
import pandas
import plotly.express as px

path_ = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments\015"


result_df = []
for run in os.listdir(path_):
    with open(os.path.join(path_, run, "params.json"), "r") as f:
        params = json.load(f)

    metrics_df = pandas.read_csv(os.path.join(path_, run, "internal_metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        oar = params["oars"] if isinstance(params["oars"], str) else "+".join(params["oars"])
        features = "+".join([k for k,v in params["features"].items() if v])
        result_df.append({"cohort": "internal", "oar": oar.split("_")[0], "features": features, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

    metrics_df = pandas.read_csv(os.path.join(path_, run, "external_metrics.csv"))
    metrics_df = metrics_df.drop(columns=["Unnamed: 0"])
    avg_metrics = metrics_df.mean().to_dict()
    for k, v in avg_metrics.items():
        oar = params["oars"] if isinstance(params["oars"], str) else "+".join(params["oars"])
        features = "+".join([k for k,v in params["features"].items() if v])
        result_df.append({"cohort": "external", "oar": oar.split("_")[0], "features": features, "metric": k, "value": v, "std": metrics_df.std().to_dict()[k]})

result_df = pandas.DataFrame(result_df)
fig = px.bar(result_df, x="features", y="value", color="metric", barmode="group", facet_col="oar", facet_row="cohort")
fig.update_layout(autosize=False, width=1600, height=900)
fig.show()

## 016

In [ ]:
import pathlib
import json
import pandas
import plotly.express as px

base = pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments\016")

df = []
params = {}
for run in base.iterdir():
    metrics = pandas.read_csv(run.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")
    metrics["name"] = run.name

    # average bootstraps
    mean = metrics.groupby(["split", "metric", "name", "step"])["value"].mean().reset_index()
    std = metrics.groupby(["split", "metric", "name", "step"])["value"].std().reset_index()

    mean = mean.rename(columns={"value": "mean"})
    mean["std"] = std["value"]

    with open(run.joinpath("params.json"), "r") as f:
        p = json.load(f)
        mean["classifier"] = p["classifier"]

    # add to df
    df.extend(mean.to_dict(orient="records"))

df = pandas.DataFrame(df)

In [ ]:
for m in df["metric"].unique():
    fig = px.line(df[df["metric"] == m], x="step", y="mean", color="name", title=m, facet_col="classifier", facet_row="split")
    fig.update_layout(height=600)
    fig.show()

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
for m in df["metric"].unique():
    g = sns.relplot(data=df[df["metric"] == m], kind="line", x="step", y="mean", hue="name", col="classifier", row="split")
    plt.subplots_adjust(top=0.9)
    plt.suptitle(m, fontsize=16)
    plt.show()

## 017

In [ ]:
import pathlib
import json
import pandas
import plotly.express as px

base = pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments\017")

df = []
params = {}
for run in base.iterdir():
    metrics = pandas.read_csv(run.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")
    metrics["name"] = run.name

    # average bootstraps
    mean = metrics.groupby(["split", "metric", "name", "step"])["value"].mean().reset_index()
    std = metrics.groupby(["split", "metric", "name", "step"])["value"].std().reset_index()

    mean = mean.rename(columns={"value": "mean"})
    mean["std"] = std["value"]

    with open(run.joinpath("params.json"), "r") as f:
        p = json.load(f)
        mean["classifier"] = p["classifier"]

    # add to df
    df.extend(mean.to_dict(orient="records"))

df = pandas.DataFrame(df)

In [ ]:
for m in df["metric"].unique():
    fig = px.line(df[df["metric"] == m], x="step", y="mean", color="name", title=m, facet_col="classifier", facet_row="split")
    fig.update_layout(height=600)
    fig.show()

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
for m in df["metric"].unique():
    g = sns.relplot(data=df[df["metric"] == m], kind="line", x="step", y="mean", hue="name", col="classifier", row="split")
    plt.subplots_adjust(top=0.9)
    plt.suptitle(m, fontsize=16)
    plt.show()

## 018

In [ ]:
import pathlib
import json
import pandas
import plotly.express as px

base = pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments\016")

df = []
params = {}
for run in base.iterdir():
    metrics = pandas.read_csv(run.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")
    metrics["name"] = run.name

    # average bootstraps
    mean = metrics.groupby(["split", "metric", "name", "step"])["value"].mean().reset_index()
    std = metrics.groupby(["split", "metric", "name", "step"])["value"].std().reset_index()

    mean = mean.rename(columns={"value": "mean"})
    mean["std"] = std["value"]

    with open(run.joinpath("params.json"), "r") as f:
        p = json.load(f)
        mean["classifier"] = p["classifier"]

    # add to df
    df.extend(mean.to_dict(orient="records"))

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
df = pandas.DataFrame(df)
for m in df["metric"].unique():
    g = sns.relplot(data=df[df["metric"] == m], kind="line", x="step", y="mean", hue="name", col="classifier", row="split")
    plt.subplots_adjust(top=0.9)
    plt.suptitle(m, fontsize=16)
    plt.show()

## 025

In [ ]:
import pathlib
import json
import pandas
import tqdm

df = []
for path_dir in pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments").glob("*"):
    try:
        if not(int(path_dir.name) in [16,25]):
            continue
    except ValueError:
        continue

    print(path_dir.name)

    for run in tqdm.tqdm(path_dir.iterdir()):
        if not(run.joinpath("train_metrics.csv").exists()):
            continue
        
        metrics = pandas.read_csv(run.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")
        metrics["name"] = run.name
        metrics["exp"] = path_.name

        with open(run.joinpath("params.json"), "r") as f:
            p = json.load(f)
            metrics["classifier"] = p["classifier"]
            metrics["lr"] = p["lr"]

        # add to df
        df.extend(metrics.to_dict(orient="records"))

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

exp_df = pandas.DataFrame(df)
exp_df = exp_df[(exp_df["metric"] == "log_loss")]

subfigs_cols = len(exp_df["lr"].unique())
fig = plt.figure(layout='constrained', figsize=(120, 60))
fig = fig.subfigures(1, subfigs_cols, wspace=0.07)

for i, lr in enumerate(exp_df["lr"].unique()):
    lr_df = exp_df[(exp_df["lr"] == lr)]

    cols = len(lr_df["classifier"].unique())
    rows = max([len(lr_df[lr_df["classifier"] == classifier]["name"].unique()) for classifier in lr_df["classifier"].unique()])
    axes = fig[i].subplots(rows, cols)

    for c, classifier in enumerate(lr_df["classifier"].unique()):
        for r, name in enumerate(lr_df[lr_df["classifier"] == classifier]["name"].unique()):
            data = lr_df[(lr_df["classifier"] == classifier) & (lr_df["name"] == name)]
            data = data.drop(columns="bootstrap")
            sns.lineplot(ax=axes[r,c], data=data, x="step", y="value", hue="split")
            axes[r,c].set_title(f"{exp} | {lr:.0E} | {name[:5]} | {classifier}")
plt.savefig(os.path.join(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\025.png"))

## 026

In [ ]:
import pathlib
import json
import pandas
import tqdm

df = []
for path_dir in pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments").glob("*"):
    try:
        if not(int(path_dir.name) in [16,26]):
            continue
    except ValueError:
        continue

    print(path_dir.name)
    for run in tqdm.tqdm(path_dir.glob("*")):
        if not(run.joinpath("train_metrics.csv").exists()):
            continue
        
        metrics = pandas.read_csv(run.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")
        metrics["name"] = run.name
        metrics["exp"] = path_.name

        with open(run.joinpath("params.json"), "r") as f:
            p = json.load(f)
            metrics["classifier"] = p["classifier"]
            metrics["bsize"] = p["bsize"]

        # add to df
        df.extend(metrics.to_dict(orient="records"))

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

exp_df = pandas.DataFrame(df)
exp_df = exp_df[(exp_df["metric"] == "auc")]

cols = len(exp_df["bsize"].unique())
rows = max([len(exp_df[exp_df["bsize"] == bsize]["name"].unique()) for bsize in exp_df["bsize"].unique()])
fig, axes = plt.subplots(rows, cols, figsize=(50, 80))

for c, bsize in enumerate(exp_df["bsize"].unique()):
    for r, name in enumerate(exp_df[exp_df["bsize"] == bsize]["name"].unique()):
        data = exp_df[(exp_df["bsize"] == bsize) & (exp_df["name"] == name)]
        data = data.drop(columns="bootstrap")
        sns.lineplot(ax=axes[r,c], data=data, x="step", y="value", hue="split")
        classifier = list(data["classifier"].unique())[0]
        axes[r,c].set_title(f"{exp} | {bsize} | {name[:5]} | {classifier}")
plt.subplots_adjust(hspace=0.3)
fig.set_size_inches(18, 100)
plt.show()

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

exp_df = pandas.DataFrame(df)
exp_df = exp_df[(exp_df["metric"] == "log_loss")]

subfigs_cols = len(exp_df["bsize"].unique())
fig = plt.figure(layout='constrained', figsize=(50, 60))
fig = fig.subfigures(1, subfigs_cols, wspace=0.07)

for i, bsize in enumerate(exp_df["bsize"].unique()):
    bsize_df = exp_df[(exp_df["bsize"] == bsize)]

    cols = len(bsize_df["classifier"].unique())
    rows = max([len(bsize_df[bsize_df["classifier"] == classifier]["name"].unique()) for classifier in bsize_df["classifier"].unique()])
    axes = fig[i].subplots(rows, cols)

    for c, classifier in enumerate(bsize_df["classifier"].unique()):
        for r, name in enumerate(bsize_df[bsize_df["classifier"] == classifier]["name"].unique()):
            data = bsize_df[(bsize_df["classifier"] == classifier) & (bsize_df["name"] == name)]
            data = data.drop(columns="bootstrap")
            sns.lineplot(ax=axes[r,c], data=data, x="step", y="value", hue="split")
            axes[r,c].set_title(f"{exp} | {bsize} | {name[:5]} | {classifier}")
# plt.subplots_adjust(hspace=0.3)
# plt.show()
plt.savefig(os.path.join(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\fig.png"))

## 016-026

In [ ]:
import pathlib
import json
import pandas
import tqdm

base_path = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments"
df = []
for path_ in pathlib.Path(base_path).glob("*"):
    print(path_.name)
    
    try:
        if int(path_.name) < 16 or int(path_.name) == 21:
            continue
    except ValueError:
        continue

    for run in tqdm.tqdm(path_.iterdir()):
        if not(run.joinpath("train_metrics.csv").exists()):
            continue
        
        metrics = pandas.read_csv(run.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")
        metrics["name"] = run.name
        metrics["exp"] = path_.name

        with open(run.joinpath("params.json"), "r") as f:
            p = json.load(f)
            metrics["classifier"] = p["classifier"]

        # add to df
        df.extend(metrics.to_dict(orient="records"))

In [ ]:
import seaborn as sns
import tqdm
import os
from matplotlib import pyplot as plt

AVERAGE_BOOTSTRAP = True

dff = pandas.DataFrame(df)

for metric in dff["metric"].unique():
    print(metric)
    for exp in tqdm.tqdm(dff["exp"].unique()):
        exp_df = dff[(dff["exp"] == exp) & (dff["metric"] == metric)]
        print(exp)

        cols = len(exp_df["classifier"].unique())
        rows = max([len(exp_df[exp_df["classifier"] == c]["name"].unique()) for c in exp_df["classifier"].unique()])
        fig, axes = plt.subplots(rows, cols, figsize=(30, 40))
        
        for c, classifier in enumerate(exp_df["classifier"].unique()):
            for r, name in enumerate(exp_df[exp_df["classifier"] == classifier]["name"].unique()):
                data = exp_df[(exp_df["classifier"] == classifier) & (exp_df["name"] == name)]
                if AVERAGE_BOOTSTRAP:   # average bootstraps                    
                    # mean = data.groupby(["split", "metric", "name", "step", "classifier"])["value"].mean().reset_index()
                    # std = data.groupby(["split", "metric", "name", "step", "classifier"])["value"].std().reset_index()
                    # mean["std"] = std["value"]
                    
                    data = data.drop(columns="bootstrap")
                    if cols > 1:
                        sns.lineplot(ax=axes[r,c], data=data, x="step", y="value", hue="split")
                    else:
                        sns.lineplot(ax=axes[r], data=data, x="step", y="value", hue="split")
                else:
                    if cols > 1:
                        sns.lineplot(ax=axes[r,c], data=data, x="step", y="value", hue="bootstrap")
                    else:
                        sns.lineplot(ax=axes[r], data=data, x="step", y="value", hue="bootstrap")
                
                if cols > 1:
                    axes[r,c].set_title(f"{exp} | {classifier} | {name}")
                else:
                    axes[r].set_title(f"{exp} | {classifier} | {name}")
        plt.subplots_adjust(hspace=0.3)
        fig.set_size_inches(18, 100)
        # plt.show()
        plt.savefig(os.path.join(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\plots\training metrics", f"{metric}-{exp}.png"))

## 029+

In [ ]:
import pathlib
import json
import pandas
import tqdm

base_path = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments"
df = []
for path_ in pathlib.Path(base_path).glob("*"):
    print(path_.name)
    
    try:
        if int(path_.name) != 34:
            continue
    except ValueError:
        continue

    for run in tqdm.tqdm(path_.iterdir()):
        if not(run.joinpath("train_metrics.csv").exists()):
            continue
        
        metrics = pandas.read_csv(run.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")
        metrics["name"] = run.name
        metrics["exp"] = path_.name

        with open(run.joinpath("params.json"), "r") as f:
            p = json.load(f)
            metrics["classifier"] = p["classifier"]

        # add to df
        df.extend(metrics.to_dict(orient="records"))

In [ ]:
import seaborn as sns
import tqdm
import os
from matplotlib import pyplot as plt

AVERAGE_KFOLD = True

dff = pandas.DataFrame(df)

for exp in tqdm.tqdm(dff["exp"].unique()):
    exp_df = dff[(dff["exp"] == exp) & (dff["metric"] == "log_loss")]
    print(exp)

    cols = len(exp_df["classifier"].unique())
    rows = max([len(exp_df[exp_df["classifier"] == c]["name"].unique()) for c in exp_df["classifier"].unique()])
    fig, axes = plt.subplots(rows, cols, figsize=(30, 40))
    
    for c, classifier in enumerate(exp_df["classifier"].unique()):
        for r, name in enumerate(exp_df[exp_df["classifier"] == classifier]["name"].unique()):
            data = exp_df[(exp_df["classifier"] == classifier) & (exp_df["name"] == name)]
            if AVERAGE_KFOLD:   # average kfolds                  
                data = data.drop(columns="kfold")
                if cols > 1:
                    sns.lineplot(ax=axes[r,c], data=data, x="step", y="value", hue="split")
                else:
                    sns.lineplot(ax=axes[r], data=data, x="step", y="value", hue="split")
            else:
                if cols > 1:
                    sns.lineplot(ax=axes[r,c], data=data, x="step", y="value", hue="kfold")
                else:
                    sns.lineplot(ax=axes[r], data=data, x="step", y="value", hue="kfold")
            
            if cols > 1:
                axes[r,c].set_title(f"{exp} | {classifier} | {name[:15]}")
            else:
                axes[r].set_title(f"{exp} | {classifier} | {name[:15]}")
    plt.subplots_adjust(hspace=0.3)
    fig.set_size_inches(18, 10)
    plt.savefig(os.path.join(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp", f"log_loss-{exp}.png"))

## 029-033

In [ ]:
import pathlib
import json
import pandas
import tqdm

base_path = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments"
df = []
for path_ in pathlib.Path(base_path).glob("*"):
    print(path_.name)
    
    try:
        if int(path_.name) != 39:
            continue
    except ValueError:
        continue

    for run in tqdm.tqdm(path_.iterdir()):
        if not(run.joinpath("train_metrics.csv").exists()):
            continue
        
        metrics = pandas.read_csv(run.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")
        metrics["name"] = run.name
        metrics["exp"] = path_.name

        with open(run.joinpath("params.json"), "r") as f:
            p = json.load(f)
            metrics["classifier"] = p["classifier"]

        # add to df
        df.extend(metrics.to_dict(orient="records"))

In [ ]:
import seaborn as sns
import tqdm
import os
from matplotlib import pyplot as plt

AVERAGE_KFOLD = True

dff = pandas.DataFrame(df)

for exp in tqdm.tqdm(dff["exp"].unique()):
    exp_df = dff[(dff["exp"] == exp) & (dff["metric"] == "log_loss")]
    print(exp)

    cols = len(exp_df["classifier"].unique())
    rows = max([len(exp_df[exp_df["classifier"] == c]["name"].unique()) for c in exp_df["classifier"].unique()])
    fig, axes = plt.subplots(rows, cols)
    
    for c, classifier in enumerate(exp_df["classifier"].unique()):
        for r, name in enumerate(exp_df[exp_df["classifier"] == classifier]["name"].unique()):
            data = exp_df[(exp_df["classifier"] == classifier) & (exp_df["name"] == name)]
            if AVERAGE_KFOLD:   # average kfolds                  
                data = data.drop(columns="kfold")
                if cols > 1:
                    sns.lineplot(ax=axes[r,c], data=data, x="step", y="value", hue="split")
                else:
                    sns.lineplot(ax=axes[r], data=data, x="step", y="value", hue="split")
            else:
                if cols > 1:
                    sns.lineplot(ax=axes[r,c], data=data, x="step", y="value", hue="kfold")
                else:
                    sns.lineplot(ax=axes[r], data=data, x="step", y="value", hue="kfold")
            
            if cols > 1:
                axes[r,c].set_title(f"{exp} | {classifier} | {name[:15]}")
            else:
                axes[r].set_title(f"{exp} | {classifier} | {name[:15]}")
    plt.subplots_adjust(hspace=0.3)
    fig.set_size_inches(8*cols, 6*rows)
    plt.savefig(os.path.join(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp", f"log_loss-{exp}.png"))

## model prediction distribution


### synthetic data

In [ ]:
import seaborn as sns
from itertools import chain
import tqdm
import torch
import pathlib
import json
from matplotlib import pyplot as plt

from classifiers.classifiers import Attention, Linear, Concat

device = "cuda:3"
N = 10000

df = []
for path_ in pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments").glob("*"):
    if not(int(path_.name) in (16,19,20)):
        continue
    print(path_)
    for model_dir in tqdm.tqdm(pathlib.Path(path_).iterdir()):
        with open(model_dir.joinpath("params.json"), "r") as f:
            params = json.load(f)

        match params["classifier"]:
            case "linear":
                n_dim = list(chain.from_iterable([v.values() for v in params["dims"].values()]))[0]
                model = Linear(n_dim)
            case "concat":
                model = Concat(**params)
            case "attention":
                model = Attention(**params)
        
        model.load(model_dir.joinpath("best_checkpoint_0.pt"))
        model = model.eval().to(device)

        # N(0,1)
        d = {m: 
                {f: torch.distributions.normal.Normal(torch.zeros(v, dtype=torch.float32),  torch.ones(v, dtype=torch.float32)) for f,v in params["dims"][m].items()} 
                for m in params["dims"].keys()}
        
        with torch.no_grad():
            x = {m: 
                    {f: d[m][f].sample(sample_shape=(N,)).to(device, torch.float32) for f in d[m].keys()} 
                    for m in d.keys()}
            y_pred = torch.sigmoid(model(x).detach().cpu().flatten())
            df.extend([{"p": p, "name": model_dir.name, "classifier": params['classifier'], "exp": model_dir.parent.name} for p in y_pred.tolist()])

        del model

In [ ]:
import pandas
dff = pandas.DataFrame(df)
rows = len(dff["classifier"].unique())
cols = len(dff["exp"].unique())

fig, axes = plt.subplots(rows, cols, figsize=(30, 20))
for i, classifier in enumerate(dff["classifier"].unique()):
    for j, exp in enumerate(dff["exp"].unique()):
        g = sns.kdeplot(ax=axes[i,j], data=dff[(dff["classifier"] == classifier) & (dff["exp"] == exp)], x="p", hue="name")
        axes[i,j].legend([],[], frameon=False)
        axes[i,j].set_title(f"{exp} | {classifier}")
        axes[i,j].set_xlim(0., 1.)
        axes[i,j].axvline(0.5, color="black", linestyle="--")
plt.show()

### true data

In [ ]:
from dataloader import ARTIX, QINHEADNECK
from dataloader import HECKTOR, HeadNeckCTAtlas, HeadNeckPETCT, OropharyngealRadiomicsOutcomes, RADCURE
import pandas
import tqdm

base_path = "./"
features = pandas.DataFrame([])
label = pandas.DataFrame([])
for loader in tqdm.tqdm([ARTIX, QINHEADNECK, HECKTOR, HeadNeckCTAtlas, HeadNeckPETCT, OropharyngealRadiomicsOutcomes, RADCURE], ncols=100):
    loader = loader(None)
    x, y = loader.build_dataset(base_path)
    features = pandas.concat((features, x))
    label = pandas.concat((label, y))
print(features)
print(label)

In [ ]:
import torch.nn.functional as F
import torch
from classifiers.classifiers import Attention, Linear, Concat, Normalizer
import pickle
import json
import pathlib
from itertools import chain

from experiments import INTERNAL_CENTERS, EXTERNAL_CENTERS, DataLoader, send_to_device

device = "cuda:3"

df = []
for path_ in pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments").glob("*"):
    print(path_)

    try:
        if not(int(path_.name) in (16,19,20,23)):
            continue
    except ValueError:
        continue
    
    for model_dir in tqdm.tqdm(pathlib.Path(path_).iterdir()):
        with open(model_dir.joinpath("params.json"), "r") as f:
            params = json.load(f)

        match params["classifier"]:
            case "linear":
                n_dim = list(chain.from_iterable([v.values() for v in params["dims"].values()]))[0]
                model = Linear(n_dim)
            case "concat":
                model = Concat(**params)
            case "attention":
                model = Attention(**params)

        normalizer = Normalizer(params["normalizer"])
        with open(model_dir.joinpath("normalizer_0.pickle"), "rb") as f:
            normalizer.set_params(pickle.load(f))
        
        model.load(model_dir.joinpath("best_checkpoint_0.pt"))
        model = model.eval().to(device)

        data_loader = DataLoader(base_path="./", X=features, Y=label)
        data_loader.prepare_data(params, params["task"])
        X, Y, *_ = data_loader.split(INTERNAL_CENTERS, EXTERNAL_CENTERS)
        X = normalizer.transform(X)
        data_loader = DataLoader(base_path="./", X=X, Y=Y)
        
        for center, center_loader in data_loader.split_per_center():
            for batch in center_loader.batch_iterator(32):
                if batch == StopIteration:
                    break
                x, y = batch
                with torch.no_grad():
                    pred_proba = model(send_to_device(x, device)).to("cpu")
                    pred_proba = F.sigmoid(pred_proba).flatten()

                for y_pred, y_true in zip(pred_proba, y):
                    df.append({"name": model_dir.name, "classifier": params['classifier'], "exp": model_dir.parent.name, "y_pred": float(y_pred), "y_true": int(y_true.item()), "center": center})
        del model

In [ ]:
import pandas
import os
import seaborn as sns
from matplotlib import pyplot as plt

dff = pandas.DataFrame(df)
for exp in dff["exp"].unique():
    print(exp)
    exp_df = dff[dff["exp"] == exp]
    cols = len(exp_df["classifier"].unique())
    rows = max([len(exp_df[exp_df["classifier"] == c]["name"].unique()) for c in exp_df["classifier"].unique()])
    fig, axes = plt.subplots(rows, cols, figsize=(30, 50))

    for j, classifier in enumerate(exp_df["classifier"].unique()):
        for i, name in enumerate(exp_df[exp_df["classifier"] == classifier]["name"].unique()):
            g = sns.kdeplot(ax=axes[i,j], data=exp_df[(exp_df["classifier"] == classifier) & (exp_df["name"] == name)], x="y_pred", hue="y_true")
            axes[i,j].set_title(f"{exp} | {classifier} | {name}")
            axes[i,j].set_xlim(0., 1.)
            axes[i,j].axvline(0.5, color="black", linestyle="--")
    # plt.show()
    plt.savefig(os.path.join(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\plots\prediction distribution\train data", f"{exp}.png"))

In [ ]:
dff = pandas.DataFrame(df)
for exp in dff["exp"].unique():
    print(exp)
    exp_df = dff[dff["exp"] == exp]
    cols = 2 # y=0 and y=1
    rows = len(exp_df["name"].unique())
    fig, axes = plt.subplots(rows, cols, figsize=(30, 100))
    for r, name in enumerate(exp_df["name"].unique()):
        classifier = exp_df[exp_df["name"] == name]["classifier"].unique()[0]
        for y in exp_df[exp_df["name"] == name]["y_true"].unique():
            g = sns.kdeplot(ax=axes[r,y], data=exp_df[(exp_df["name"] == name) & (exp_df["y_true"] == y)], x="y_pred", hue="center", common_norm=False)
            axes[r,y].set_title(f"{exp} | {name} | {classifier} | y={y}")
            axes[r,y].set_xlim(0., 1.)
            axes[r,y].axvline(0.5, color="black", linestyle="--")
    plt.show()

## compile and save to excel

In [ ]:
import glob
import tqdm
import pandas
import pathlib
import json

out_path = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\combined_metrics_test_039.xlsx"
all_df = {}
for path_ in pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments").glob("*"):
    try:
        if int(path_.name) != 39:
            continue
    except ValueError:
        continue

    print(path_)
    df = []
    for model_dir in tqdm.tqdm(path_.iterdir()):
        if not(model_dir.joinpath("train_metrics.csv").exists()):
            continue

        # get best validation loss step
        metrics = pandas.read_csv(model_dir.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")   # load model metrics
        metrics = metrics[(metrics["split"] == "valid") & (metrics["metric"] == "log_loss")]   # keep only validation loss
        mean = metrics.groupby(["split", "metric", "step"])["value"].mean().reset_index()   # average bootstraps
        min_val_loss_step = mean[["value", "step"]]
        step = min_val_loss_step.loc[min_val_loss_step["value"].idxmin()]["step"]

        # load test metrics        
        test_metrics = pandas.read_csv(model_dir.joinpath("test_metrics.csv")).drop(columns="Unnamed: 0")

        # select metrics at step
        test_metrics = test_metrics[test_metrics["step"] == step]

        # average over bootstraps
        # test_metrics = test_metrics.groupby(["split", "metric", "step", "center"])["value"].mean().reset_index()
        test_metrics = test_metrics.groupby(["split", "metric", "step"])["value"].mean().reset_index()

        # add name and classifier type
        with open(model_dir.joinpath("params.json"), "r") as f:
            p = json.load(f)
            test_metrics["name"] = model_dir.name
            test_metrics["classifier"] = p['classifier']
            if "image" in p["dims"].keys():
                test_metrics["FM-I"] = len(p["dims"]["image"])
            else:
                test_metrics["FM-I"] = None
            if "clinical" in p["dims"].keys():
                test_metrics["FM-C"] = p["dims"]["clinical"]
            else:
                test_metrics["FM-C"] = None

        # add to df
        df.extend(test_metrics.to_dict(orient="records"))
    if df:
        # df = pandas.DataFrame(df).pivot(index=["classifier", "FM-I", "FM-C", "name"], columns=["center", "metric"], values="value").sort_values(by="center", axis=1)
        df = pandas.DataFrame(df).pivot(index=["classifier", "FM-I", "FM-C", "name"], columns=["metric"], values="value").sort_values(by="metric", axis=1)
        all_df.update({path_.name: df})

with pandas.ExcelWriter(out_path) as writer:
    for k, df in all_df.items():
        df.to_excel(writer, sheet_name=k)

### batch size

In [ ]:
import glob
import tqdm
import pandas
import pathlib
import json

out_path = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\combined_metrics_test_bs.xlsx"
df = []
for path_ in pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments").glob("*"):
    try:
        if not int(path_.name) == 31:
            continue
    except ValueError:
        continue

    print(path_)
    for model_dir in tqdm.tqdm(path_.iterdir()):
        if not(model_dir.joinpath("train_metrics.csv").exists()):
            continue

        # get best validation loss step
        metrics = pandas.read_csv(model_dir.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")   # load model metrics
        metrics = metrics[(metrics["split"] == "valid") & (metrics["metric"] == "log_loss")]   # keep only validation loss
        mean = metrics.groupby(["split", "metric", "step"])["value"].mean().reset_index()   # average bootstraps
        min_val_loss_step = mean[["value", "step"]]
        step = min_val_loss_step.loc[min_val_loss_step["value"].idxmin()]["step"]

        # load test metrics        
        test_metrics = pandas.read_csv(model_dir.joinpath("test_metrics.csv")).drop(columns="Unnamed: 0")

        # select metrics at step
        test_metrics = test_metrics[test_metrics["step"] == step]

        # average over bootstraps
        test_metrics = test_metrics.groupby(["split", "metric", "step"])["value"].mean().reset_index()

        # add name and classifier type
        with open(model_dir.joinpath("params.json"), "r") as f:
            p = json.load(f)
            test_metrics["name"] = model_dir.name
            test_metrics["classifier"] = p['classifier']
            test_metrics["bsize"] = p['bsize']
            if "image" in p["dims"].keys():
                test_metrics["FM-I"] = len(p["dims"]["image"])

            if "clinical" in p["dims"].keys():
                test_metrics["FM-C"] = p["dims"]["clinical"]

        # add to df
        df.extend(test_metrics.to_dict(orient="records"))

df = pandas.DataFrame(df).pivot(index=["bsize", "classifier", "FM-I", "FM-C", "name"], columns=["metric"], values="value")
df.to_excel(out_path)

### learning rate

In [ ]:
import glob
import tqdm
import pandas
import pathlib
import json

out_path = r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\combined_metrics_test_lr.xlsx"
df = []
for path_ in pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments").glob("*"):
    try:
        if not int(path_.name) == 31:
            continue
    except ValueError:
        continue

    print(path_)
    for model_dir in tqdm.tqdm(path_.iterdir()):
        if not(model_dir.joinpath("train_metrics.csv").exists()):
            continue

        # get best validation loss step
        metrics = pandas.read_csv(model_dir.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")   # load model metrics
        metrics = metrics[(metrics["split"] == "valid") & (metrics["metric"] == "log_loss")]   # keep only validation loss
        mean = metrics.groupby(["split", "metric", "step"])["value"].mean().reset_index()   # average bootstraps
        min_val_loss_step = mean[["value", "step"]]
        step = min_val_loss_step.loc[min_val_loss_step["value"].idxmin()]["step"]

        # load test metrics        
        test_metrics = pandas.read_csv(model_dir.joinpath("test_metrics.csv")).drop(columns="Unnamed: 0")

        # select metrics at step
        test_metrics = test_metrics[test_metrics["step"] == step]

        # average over bootstraps
        test_metrics = test_metrics.groupby(["split", "metric", "step"])["value"].mean().reset_index()

        # add name and classifier type
        with open(model_dir.joinpath("params.json"), "r") as f:
            p = json.load(f)
            test_metrics["name"] = model_dir.name
            test_metrics["classifier"] = p['classifier']
            test_metrics["lr"] = p['lr']
            if "image" in p["dims"].keys():
                test_metrics["FM-I"] = len(p["dims"]["image"])
            if "clinical" in p["dims"].keys():
                test_metrics["FM-C"] = p["dims"]["clinical"]

        # add to df
        df.extend(test_metrics.to_dict(orient="records"))

df = pandas.DataFrame(df).pivot(index=["lr", "classifier", "FM-I", "FM-C", "name"], columns=["metric"], values="value")
df.to_excel(out_path)

## metrics bars

In [ ]:
import glob
import tqdm
import pandas
import pathlib
import json

df = []
for path_ in pathlib.Path(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\experiments").glob("*"):
    try:
        if int(path_.name) != 39:
            continue
    except ValueError:
        continue

    print(path_)
    for model_dir in tqdm.tqdm(path_.iterdir()):
        if not(model_dir.joinpath("train_metrics.csv").exists()):
            continue

        # get best validation loss step
        metrics = pandas.read_csv(model_dir.joinpath("train_metrics.csv")).drop(columns="Unnamed: 0")   # load model metrics
        metrics = metrics[(metrics["split"] == "valid") & (metrics["metric"] == "log_loss")]   # keep only validation loss
        mean = metrics.groupby(["split", "metric", "step"])["value"].mean().reset_index()   # average bootstraps
        min_val_loss_step = mean[["value", "step"]]
        step = min_val_loss_step.loc[min_val_loss_step["value"].idxmin()]["step"]

        # load test metrics        
        test_metrics = pandas.read_csv(model_dir.joinpath("test_metrics.csv")).drop(columns="Unnamed: 0")

        # select metrics at step
        test_metrics = test_metrics[test_metrics["step"] == step]

        # average over bootstraps
        # test_metrics = test_metrics.groupby(["split", "metric", "step"])["value"].mean().reset_index()
        test_metrics = test_metrics.drop(columns=["split", "step"])

        # add name and classifier type
        with open(model_dir.joinpath("params.json"), "r") as f:
            p = json.load(f)
            test_metrics["name"] = model_dir.name
            test_metrics["classifier"] = p['classifier']
            
            if "image" in p["dims"].keys() and not("clinical" in p["dims"].keys()):
                test_metrics["modality"] = "image"
            elif not("image" in p["dims"].keys()) and "clinical" in p["dims"].keys():
                test_metrics
            else:
                test_metrics["modality"] = "both"

            test_metrics = test_metrics[~test_metrics["metric"].isin(["acc", "log_loss"])]

        # add to df
        df.extend(test_metrics.to_dict(orient="records"))

df = pandas.DataFrame(df)
print(df)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Setup Data Identifiers (Same as before)
df['run_id'] = df.groupby(['metric', 'modality', 'classifier', 'kfold']).cumcount() + 1
df['modality_run'] = df['modality'] + " (R" + df['run_id'].astype(str) + ")"
df = df.sort_values(['metric', 'modality', 'run_id'])

sns.set_theme(style="whitegrid")

# 2. Create the Plot
g = sns.catplot(
    data=df, 
    kind="bar",
    x="modality_run", 
    y="value", 
    hue="classifier",
    row="metric",
    errorbar="sd",     
    capsize=0.1,       
    height=4, 
    aspect=3,
    palette="viridis",
    dodge=True
)

# 3. Add Labels with extra padding to avoid error bar overlap
for ax in g.axes.flat:
    for container in ax.containers:
        # Increase padding to 12 or 15 to clear the 'sd' whiskers
        ax.bar_label(container, fmt='%.2f', padding=15, fontsize=8, fontweight='bold')

# 4. Critical Step: Expand Y-limit
# We increase the limit by 30% to ensure the floating text stays inside the plot area
for ax in g.axes.flat:
    curr_ylim = ax.get_ylim()
    ax.set_ylim(0, curr_ylim[1] * 1.01)

# 5. Final Formatting
g.set_axis_labels("Modality & Run ID", "Mean Score (K-Folds ± SD)")
g.set_titles(row_template="{row_name}")
g.tight_layout()

plt.savefig(r"C:\Users\bilel.guetarni\Desktop\workspace\SEQ-RT\tmp\metrics-039.png")
plt.show()